Load in the proper libraries and dataset. We will be using packages from last week too!

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Load the Uber Data
df = pd.read_csv("UberDataset Current.csv")

1. Convert START_DATE into a datetime object.

Aggregate the number of rides that exist in any given date's hour (ride_count). Keep the features is_rush_hour, is_weekend, and temperature.

Convert is_rush_hour and is_weekend into 0/1 to represent booleans.

In [ ]:
df['START_DATE'] = pd.to_datetime(df['START_DATE'])
df['date'] = df['START_DATE'].dt.date

# 2. Feature Engineering: Grouping data to create our Target Variable (Y)
# We need to know "how many rides" occurred per hour to predict driver demand.
df_grouped = df.groupby(['date', 'hour']).agg(
    ride_count=('START_DATE', 'count'),          # rides in this hour
    is_rush_hour=('is_rush_hour', 'first'),      # rush hour?
    is_weekend=('is_weekend', 'first'),          # Weekend?
    TEMPERATURE=('TEMPERATURE', 'mean')          # Temperature
).reset_index().dropna()

# Convert into int boolean
df_grouped['is_rush_hour'] = df_grouped['is_rush_hour'].astype(int)
df_grouped['is_weekend'] = df_grouped['is_weekend'].astype(int)

df_grouped.head(50)

,date,hour,ride_count,is_rush_hour,is_weekend,TEMPERATURE
0,2016-01-01,21,1,0,0,80.00
1,2016-01-02,1,1,0,0,75.50
2,2016-01-02,20,1,0,0,71.80
3,2016-01-05,17,1,1,0,61.30
4,2016-01-06,14,1,0,0,67.40
5,2016-01-06,17,2,1,0,70.90
6,2016-01-07,13,1,0,0,48.20
7,2016-01-10,8,1,1,1,51.70
8,2016-01-10,12,1,0,1,84.90
9,2016-01-10,15,1,0,1,52.90


2. Using the new features from above, create sub-dataframes. X should be hour, is_rush_hour, is_weekend, temperature. Y should be ride_count.

Run X and Y via train_test_split().

Train Linear Regression Model (see last week's slide).

In [ ]:
# Define Features (X) and Target (y)
X = df_grouped[['hour', 'is_rush_hour', 'is_weekend', 'TEMPERATURE']]
y = df_grouped['ride_count']

# Split the dataset: 80% Train, 20% Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the Linear Regression Model
model = LinearRegression()
model.fit(X_train, y_train)

print("Model successfully trained on 80% of the data!")

Model successfully trained on 80% of the data!


3. Building off of fitting, call two prediction sets. One .predict() should deal exclusively with training data, while the other should deal with testing data.

With each respective call, calculate the mean absolute error.

Ex: "Train MAE: 0.23 rides"

In [ ]:
# Make predictions on both sets
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Calculate MAE
train_mae = mean_absolute_error(y_train, y_pred_train)
test_mae = mean_absolute_error(y_test, y_pred_test)

print(f"Train MAE: {train_mae:.2f} rides")
print(f"Test MAE: {test_mae:.2f} rides")

Train MAE: 0.26 rides
Test MAE: 0.23 rides


4. Calculate the residuals on Y.

Find the Standard Error of the residuals.

Calculate the 95% Confidence Margin.


In [ ]:
# 1. Calculate residuals (Difference between Actual and Predicted on the Test Set)
residuals = y_test - y_pred_test

# 2. Find the Standard Error
std_error = np.std(residuals)

# 3. Calculate the 95% Confidence Margin
confidence_margin = 1.96 * std_error

print(f"95% Confidence Margin: +/- {confidence_margin:.2f} rides")

95% Confidence Margin: +/- 0.62 rides


5. Get the first prediction from your test set's prediction. Calculate upper and lower bound.



In [ ]:
# Get the first prediction from y_pred_test
example_prediction = y_pred_test[0]

# Calculate lower and upper bounds
lower_bound = example_prediction - confidence_margin
upper_bound = example_prediction + confidence_margin

print(f"Model Prediction: {example_prediction:.1f} rides")
print(f"Confidence Range: {lower_bound:.1f} to {upper_bound:.1f} rides")
print("\nStakeholder Pitch:")
print(f"\"Based on weather and time data, we anticipate needing {example_prediction:.0f} drivers, but we advise preparing for a potential range of {lower_bound:.0f} to {upper_bound:.0f} drivers.\"")

Model Prediction: 1.1 rides
Confidence Range: 0.5 to 1.7 rides

Stakeholder Pitch:
"Based on weather and time data, we anticipate needing 1 drivers, but we advise preparing for a potential range of 0 to 2 drivers."
